# 09 - Live scoring of currently-confirmed bookings

Scores every booking that is **Confirmed right now** (open, not yet resolved) with
each trained model - LogReg (01), XGBoost (02), HistGB (03) and the hazard model (08) -
and writes an **Excel** file with the id, the relevant features and one
predicted-cancel-probability column per model, plus the scoring timestamp. Purpose:
park it, wait a few weeks, and manually check whether the flagged bookings actually
cancel. Requires the model notebooks (01/02/03/08) to have been run + persisted.

## 0 - Setup

In [ ]:
from __future__ import annotations
import sys, time
from pathlib import Path
_here = Path.cwd().resolve()
while not (_here/"pyproject.toml").exists():
    if _here==_here.parent: raise RuntimeError("project root not found")
    _here=_here.parent
if str(_here) not in sys.path: sys.path.insert(0, str(_here))
import numpy as np, pandas as pd
import plotly.graph_objects as go, plotly.io as pio
from src import load_reservations, load_clean_reservations, color, data_dir, tables_dir
from src.features import load_feature_roster, family_feature_lists
import src.scoring as sc, src.hazard as HZ
pio.templates.default="plotly_white"
BRAND={n: color(n) for n in ["blue","orange","green","purple","red"]}
SCORED_AT = pd.Timestamp.now("UTC")
print("scoring timestamp (UTC):", SCORED_AT)

## 1 - Load currently-confirmed bookings

`status == "Confirmed"` = open bookings that will resolve over the coming weeks.
`force_refresh=True` in the loader pulls fresh from BigQuery; here we read the cache.

In [ ]:
raw = load_reservations()                     # add force_refresh=True for a live pull
conf = raw[raw["status"].astype("string") == "Confirmed"].copy()
if "id" not in conf.columns:
    conf["id"] = np.arange(len(conf))         # fallback key (raw usually carries `id`)
arr = pd.to_datetime(conf["arrival"], utc=True); cre = pd.to_datetime(conf["created"], utc=True)
conf["lead_days"] = (arr - cre) / pd.Timedelta(days=1)
conf["days_to_arrival"] = (arr - SCORED_AT) / pd.Timedelta(days=1)
print(f"confirmed bookings: {len(conf):,}")
print(f"  arrival range : {arr.min()} -> {arr.max()}")
print(f"  days-to-arrival: median {conf['days_to_arrival'].median():.0f}  "
      f"(within 14d: {(conf['days_to_arrival']<=14).sum():,})")
display(conf.groupby(conf['arrival'].dt.to_period('M').astype(str)).size()
            .rename('n_confirmed').to_frame().tail(12))

## 2 - Build features (serving parity)

`src.scoring.build_features` reproduces 00's engineering (incl. the linear `_log`
twins) so every model family is scoreable; `apply_scoring_bounds` drops rows whose
essential features are impossible to compute (but never filters on status).

In [ ]:
roster = load_feature_roster()
feat = sc.build_features(conf)                # dynamic features are point-in-time (now)
feat = sc.apply_scoring_bounds(feat)
print(f"scoreable bookings after bounds: {len(feat):,}")
need = roster["numeric"] + roster["categorical"]
missing = [c for c in need if c not in feat.columns]
assert not missing, f"build_features missing roster features: {missing}"
print("all roster features present ->", not missing)

## 3 - Score with every trained model

Each static model gets its family view (LogReg -> linear/log twins; trees -> raw);
the hazard model scores the survival product over each booking's remaining
days-until-arrival. Missing models are skipped with a warning.

In [ ]:
out = feat.copy()
prob_cols = []
for m, family in [("logreg", "linear"), ("xgboost", "tree"), ("histgb", "tree")]:
    try:
        pipe = sc.load_model(m)
        num, cat = family_feature_lists(roster, family)
        out[f"p_{m}"] = pipe.predict_proba(out[num + cat])[:, 1]
        prob_cols.append(f"p_{m}")
        print(f"  scored {m}")
    except FileNotFoundError:
        print(f"  {m} not trained/persisted yet - skipped (run notebook first)")

try:
    hz = HZ.load_hazard()
    b = out.copy(); b["lead"] = b["lead_time_days"]
    b[HZ.AXIS] = b["days_until_arrival"].clip(lower=1)          # remaining days, live
    out["p_hazard"] = HZ.score_upcoming_hazard(hz, b)
    prob_cols.append("p_hazard")
    print("  scored hazard")
except FileNotFoundError:
    print("  hazard not trained/persisted yet - skipped")

assert prob_cols, "no models available - run 01/02/03/08 first"
if len(prob_cols) > 1:
    out["p_ensemble"] = out[prob_cols].mean(axis=1)            # simple mean of available models
    prob_cols_disp = prob_cols + ["p_ensemble"]
else:
    prob_cols_disp = prob_cols
print("probability columns:", prob_cols_disp)

## 4 - Prediction summary

In [ ]:
display(out[prob_cols_disp].describe().T.round(4))
fig = go.Figure()
for c in prob_cols_disp:
    fig.add_histogram(x=out[c], nbinsx=50, name=c, opacity=0.6)
fig.update_layout(barmode="overlay", title="Predicted cancel-probability distribution by model",
                  xaxis_title="P(cancel before arrival)", yaxis_title="bookings")
fig.show()

# agreement: pairwise correlation of the model probabilities
if len(prob_cols) > 1:
    display(out[prob_cols].corr().round(3))

## 5 - Excel export

id + key features + one probability column per model + the scoring timestamp, so you
can revisit in a few weeks and mark which bookings actually cancelled.

In [ ]:
KEEP = [c for c in ["id", "property_name", "channelCode", "guaranteeType",
                    "ratePlan_category", "arrival", "created", "lead_days",
                    "days_to_arrival", "gross_amount", "los_nights"] if c in out.columns]
export = out[KEEP + prob_cols_disp].copy()
export.insert(0, "scored_at_utc", SCORED_AT)
export = export.sort_values(prob_cols_disp[-1], ascending=False)

XLSX = data_dir() / f"live_scores_{SCORED_AT.strftime('%Y%m%d')}.xlsx"
export.to_excel(XLSX, index=False, sheet_name="live_scores")
print(f"exported {len(export):,} scored bookings -> {XLSX}")
display(export.head(20))